# Koddan Kasa · Levha 01 · Bu robot köpek bir metin dosyası

Videodaki iki deney, sıfırdan:

1. **15 satırlık dünya**: masa, iki küp, yerçekimi. Küplerden biri 10 kat ağır. İkisi masaya hangi **adımda** değiyor? Sonra dosyanın 2. satırına hava (`density="1.2"`) ekleyip tekrar.
2. **Boston Dynamics Spot'un dosyası** ([MuJoCo Menagerie](https://github.com/google-deepmind/mujoco_menagerie/tree/main/boston_dynamics_spot), BSD-3, © Clearpath Robotics): 179 satır. 62. satırdaki gövde kütlesini (`mass="32.86"`) 10 katına çıkarıp Spot ayakta ne kadar çöküyor, ölçüyoruz.

Hepsi **simülasyon**. Gerçek bir Spot'a hiçbir şey olmuyor; Menagerie'nin motor ayarları README'de de yazdığı gibi "not tuned".

## 1 · Kurulum (MuJoCo + sadece Spot modeli)

In [ ]:
!pip -q install mujoco mediapy
# Birebir aynı sayılar için: !pip -q install mujoco==3.17.0 mediapy
![ -d mujoco_menagerie ] || git clone -q --depth 1 --filter=blob:none --sparse https://github.com/google-deepmind/mujoco_menagerie
!cd mujoco_menagerie && git sparse-checkout set boston_dynamics_spot && ls boston_dynamics_spot

## 2 · Dünya = metin: 15 satır

Aşağıdaki yazı dünyanın **tamamı**. `timestep` yazılmadığı için MuJoCo varsayılanı 0,002 sn: saniyede 500 adım. Hava da yazılmadığı için yok (`density = 0`).
Masanın üstü 0,40 m'de, küpler 20 cm, merkezleri 0,90 m'de: 40 cm düşüş.

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")  # Colab'da ekransız çizim
import mujoco
import numpy as np

TABLE_TOP, HALF = 0.40, 0.10
XML = """<mujoco>
 <option gravity="0 0 -9.81"/>
 <worldbody>
  <geom type="plane" size="2 2 .1"/>
  <geom name="masa" type="box" size=".35 .25 .4"/>
  <body name="hafif" pos="-.13 0 .9">
   <freejoint/>
   <geom type="box" size=".1 .1 .1" mass=".1"/>
  </body>
  <body name="ağır" pos=".13 0 .9">
   <freejoint/>
   <geom type="box" size=".1 .1 .1" mass=".1"/>
  </body>
 </worldbody>
</mujoco>"""
LINES = XML.split("\n")
print(len(LINES), "satır")


def make_xml(light_kg=0.1, heavy_kg=1.0, air=False):
    lines = list(LINES)
    f = lambda kg: (f"{kg:g}"[1:] if f"{kg:g}".startswith("0.") else f"{kg:g}")
    lines[7] = lines[7].replace('mass=".1"', f'mass="{f(light_kg)}"')
    lines[11] = lines[11].replace('mass=".1"', f'mass="{f(heavy_kg)}"')
    if air:
        lines[1] = ' <option gravity="0 0 -9.81" density="1.2"/>'
    return "\n".join(lines)


def drop(xml, sim_sec=1.0):
    """Her küp için: alt yüzün masa üstüne ilk indiği fizik adımı + o anda iki küp arasındaki yükseklik farkı."""
    m = mujoco.MjModel.from_xml_string(xml)
    d = mujoco.MjData(m)
    adr = {n: m.jnt_qposadr[m.body_jntadr[m.body(n).id]] for n in ("hafif", "ağır")}
    q = [d.qpos.copy()]
    while d.time < sim_sec - 1e-9:
        mujoco.mj_step(m, d)
        q.append(d.qpos.copy())
    q = np.array(q)
    step = {n: int(np.argmax(q[:, adr[n] + 2] - HALF <= TABLE_TOP)) for n in adr}
    k = step["ağır"]
    gap_cm = (q[k, adr["hafif"] + 2] - q[k, adr["ağır"] + 2]) * 100
    return step, gap_cm

## 3 · ✏️ BURAYI DEĞİŞTİR (küpler)

Videodaki: hafif 0,1 kg, ağır 1 kg. Vakumda ikisi de **143. adımda** (0,286 sn) değiyor. Havayla (`HAVA = True`) ağır küp değdiğinde hafif olan **1,1 cm** yukarıda. Dene: `HAFIF_KG = 0.02` (kâğıt küp kadar), `0.005`…

In [ ]:
# ===================== BURAYI DEĞİŞTİR =====================
HAFIF_KG = 0.1
AGIR_KG = 1.0
HAVA = True
# ===========================================================

for hava in (False, HAVA) if HAVA else (False,):
    step, gap = drop(make_xml(HAFIF_KG, AGIR_KG, hava))
    print(f"{'hava ' if hava else 'vakum'} | hafif {step['hafif']}. adım | ağır {step['ağır']}. adım | ağır değdiğinde hafif {gap:.3f} cm yukarıda")

## 4 · Spot'un dosyası: say

Videodaki sayılar: 179 satır, 13 gövde parçası, 12 döner eklem, 12 motor; 62. satırda gövde `mass="32.86"`.

In [ ]:
SPOT_DIR = "mujoco_menagerie/boston_dynamics_spot"
SPOT_XML = open(f"{SPOT_DIR}/spot.xml").read()
lines = SPOT_XML.rstrip("\n").split("\n")
mass_line = next(i + 1 for i, l in enumerate(lines) if 'mass="32.86"' in l)


def load_spot(scale=1.0):
    xml = SPOT_XML if scale == 1 else SPOT_XML.replace('mass="32.86"', f'mass="{32.86 * scale:.4g}"')
    spec = mujoco.MjSpec.from_string(xml.replace('meshdir="assets"', f'meshdir="{os.path.abspath(SPOT_DIR)}/assets"'))
    spec.worldbody.add_geom(name="floor", type=mujoco.mjtGeom.mjGEOM_PLANE, size=[0, 0, 0.05])
    return spec.compile()


m = load_spot()
print(len(lines), "satır |", m.nbody - 1, "gövde parçası |", int((m.jnt_type == mujoco.mjtJoint.mjJNT_HINGE).sum()), "döner eklem |",
      m.nu, "motor |", f"gövde kütlesi {mass_line}. satırda | toplam {m.body_subtreemass[0]:.2f} kg | adım {m.opt.timestep} sn")

## 5 · ✏️ BURAYI DEĞİŞTİR (Spot)

Spot "home" pozundan (gövde 46 cm) bırakılıyor; Menagerie'nin pozisyon motorları (her eklemde `kp=500`, `kv=40`) aynı açıları tutmaya çalışıyor. 3 sn sonra gövde yüksekliği ölçülüyor.
Videodaki: kütle ×1 → **43,4 cm**, ×10 → **29,2 cm** (14 cm çöktü). Dene: `KUTLE_KAT = 2`, `5`, `20`…

In [ ]:
# ===================== BURAYI DEĞİŞTİR =====================
KUTLE_KAT = 10
# ===========================================================


def stand(scale, sim_sec=3.0, renderer=None, fps=30):
    m = load_spot(scale)
    d = mujoco.MjData(m)
    mujoco.mj_resetDataKeyframe(m, d, 0)
    hs, frames = [], []
    while d.time < sim_sec - 1e-9:
        if renderer is not None and len(frames) < d.time * fps:
            cam = mujoco.MjvCamera()
            cam.lookat[:] = [0, 0, 0.3]; cam.distance, cam.azimuth, cam.elevation = 2.4, 145, -10
            renderer.update_scene(d, camera=cam)
            frames.append(renderer.render())
        hs.append((d.time, d.qpos[2]))
        mujoco.mj_step(m, d)
    hs = np.array(hs)
    return hs[hs[:, 0] > sim_sec - 0.5, 1].mean() * 100, frames


h1, _ = stand(1.0)
hk, _ = stand(KUTLE_KAT)
print(f"gövde ×1: {h1:.2f} cm | gövde ×{KUTLE_KAT}: {hk:.2f} cm | çökme {h1 - hk:.2f} cm")

### Videosu (480×360, 30 fps)

In [ ]:
import mediapy as media
renderer = mujoco.Renderer(load_spot(KUTLE_KAT), height=360, width=480)
_, frames = stand(KUTLE_KAT, renderer=renderer)
renderer.close()
media.show_video(frames, fps=30)

## 6 · Bizim sayılarla karşılaştır

`public/v6/kk-01-v3/data/kk01v3.json` (MuJoCo 3.17.0). Farklı sürümde son basamaklar biraz kayabilir.

In [ ]:
OURS = {"vakum adım (hafif, ağır)": (143, 143), "hava adım (hafif, ağır)": (145, 143), "hava fark cm": 1.098,
        "spot ×1 cm": 43.39, "spot ×10 cm": 29.24}
sv, _ = drop(make_xml(0.1, 1.0, False))
sh, gh = drop(make_xml(0.1, 1.0, True))
mine = {"vakum adım (hafif, ağır)": (sv["hafif"], sv["ağır"]), "hava adım (hafif, ağır)": (sh["hafif"], sh["ağır"]), "hava fark cm": round(gh, 3),
        "spot ×1 cm": round(stand(1.0)[0], 2), "spot ×10 cm": round(stand(10.0)[0], 2)}
for k in OURS:
    print(f"{k:26s} bizim {OURS[k]!s:>12} | senin {mine[k]!s:>12}")

## 7 · Soru: küpte ağırlık fark etmedi, Spot'ta neden etti?

Cevabını yorumlara yaz. İpuçları:
- Düşen küpte yerçekimi ağır küpü 10 kat güçlü çeker, ama onu hızlandırmak da 10 kat zor (a = F / m). Spot düşmüyor, **duruyor**: neyin kuvveti ağırlığı taşıyor?
- `<position kp="500" .../>` satırı motoru bir **yay** gibi yapıyor: açı hedeften ne kadar saparsa o kadar tork. Yay 10 kat yükle ne yapar?
- `KUTLE_KAT` ile `kp`'yi birlikte değiştir (ör. `SPOT_XML.replace('kp="500"', 'kp="5000"')`): çökme ne oluyor?